# 12 — Text Generation
### Starter Notebook

**Learning objectives**
- Implement greedy decoding
- Add temperature scaling
- Implement top-k and top-p (nucleus) sampling
- Understand KV-cache and why it speeds up generation
- Compare generation strategies on the same prompt

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from src.models.transformer import TransformerLM, TransformerConfig

torch.manual_seed(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Setup — Tiny Model & Character Tokeniser

We'll train a tiny model on a short story corpus so we can see actual text output.

In [ ]:
# Tiny corpus
corpus = """
once upon a time there was a little girl named lily. she lived in a small house near the forest.
every day she walked through the trees and listened to the birds sing. one morning she found a small
rabbit sitting by the path. the rabbit looked at her with big bright eyes. lily sat down slowly so
as not to scare it. they became the best of friends. every morning lily brought carrots from her
garden and the rabbit came to meet her by the old oak tree. they played together until the sun set.
""".strip().lower()

# Character-level tokeniser
chars  = sorted(set(corpus))
VOCAB  = len(chars)
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for i, c in enumerate(chars)}
encode = lambda s: [c2i[c] for c in s]
decode = lambda ids: ''.join(i2c.get(i, '?') for i in ids)

print(f'Vocabulary size: {VOCAB} characters')
print(f'Sample encoding: {encode("hello")} → {decode(encode("hello"))}')

# Create sequences
SEQ_LEN = 48
ids     = torch.tensor(encode(corpus), dtype=torch.long)
seqs    = torch.stack([ids[i:i+SEQ_LEN] for i in range(0, len(ids)-SEQ_LEN, 4)])
print(f'Training sequences: {seqs.shape}')

In [ ]:
# Quick train on the tiny corpus
from torch.utils.data import TensorDataset, DataLoader

cfg = TransformerConfig(
    vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=3,
    d_ff=256, max_seq_len=SEQ_LEN, dropout=0.1,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
model = TransformerLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=0.01)

loader = DataLoader(TensorDataset(seqs), batch_size=16, shuffle=True)

print('Training tiny char-level model...')
losses = []
for epoch in range(50):
    for (batch,) in loader:
        batch = batch.to(device)
        x, y = batch[:, :-1], batch[:, 1:]
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB), y.view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    if (epoch+1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}  loss={losses[-1]:.4f}')

print('Done training.')

## Part A — Greedy Decoding

Always pick the token with the highest probability. Deterministic but often repetitive.

### Exercise A1

In [ ]:
@torch.no_grad()
def greedy_generate(model, prompt_ids: list, max_new_tokens: int = 50) -> list:
    """
    Generate tokens greedily (argmax at each step).

    Args:
        model      : TransformerLM
        prompt_ids : list of token ids
        max_new_tokens: how many tokens to generate

    Returns:
        Generated token ids (not including prompt)
    """
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)   # (1, prompt_len)
    generated = []

    for _ in range(max_new_tokens):
        # Clip to max_seq_len
        ids_crop = ids[:, -cfg.max_seq_len:]

        # TODO: forward pass → logits → take last position → argmax
        next_token = None   # replace with argmax of last-position logits

        if next_token is None:
            break
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)

    return generated


prompt = "once upon a time"
prompt_ids = encode(prompt)
generated = greedy_generate(model, prompt_ids, max_new_tokens=80)
print(f'Prompt    : "{prompt}"')
print(f'Generated : "{decode(generated)}"')

## Part B — Temperature Sampling

Divide logits by temperature $T$ before softmax. Lower $T$ → more confident; higher $T$ → more random.

### Exercise B1

In [ ]:
@torch.no_grad()
def temperature_generate(
    model, prompt_ids: list, max_new_tokens: int = 80, temperature: float = 1.0
) -> list:
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []

    for _ in range(max_new_tokens):
        ids_crop = ids[:, -cfg.max_seq_len:]
        logits = model(ids_crop)[:, -1, :]   # (1, vocab)

        # TODO: apply temperature, then sample from softmax distribution
        next_token = None   # replace with torch.multinomial sample

        if next_token is None:
            break
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)

    return generated


print(f'Prompt: "{prompt}"\n')
for T in [0.5, 1.0, 1.5]:
    out = temperature_generate(model, prompt_ids, temperature=T)
    print(f'T={T}:  "{decode(out)}"')
    print()

## Part C — Top-k Sampling

Keep only the top-k most probable tokens, zero out the rest, then sample.

### Exercise C1

In [ ]:
def top_k_sample(logits: torch.Tensor, k: int, temperature: float = 1.0) -> torch.Tensor:
    """
    Sample from the top-k logits.

    Args:
        logits     : (vocab_size,)
        k          : keep top-k tokens
        temperature: temperature scaling

    Returns:
        Sampled token index (scalar tensor)
    """
    logits = logits / temperature

    # TODO: zero out all but top-k values, then sample
    # Hint: use torch.topk to find the kth value, then mask with float('-inf')
    pass


@torch.no_grad()
def topk_generate(model, prompt_ids, max_new_tokens=80, k=10, temperature=1.0):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :].squeeze(0)
        next_token = top_k_sample(logits, k=k, temperature=temperature)
        if next_token is None: break
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)
    return generated


print(f'Prompt: "{prompt}"\n')
for k in [3, 10, 20]:
    out = topk_generate(model, prompt_ids, k=k, temperature=0.8)
    print(f'top-{k}:  "{decode(out)}"')
    print()

## Part D — Nucleus (Top-p) Sampling

Keep the smallest set of tokens whose cumulative probability exceeds $p$. Adapts to the distribution — more tokens when the model is uncertain.

### Exercise D1

In [ ]:
def top_p_sample(logits: torch.Tensor, p: float, temperature: float = 1.0) -> torch.Tensor:
    """
    Nucleus sampling.

    Steps:
    1. Sort probabilities in descending order
    2. Compute cumulative sum
    3. Remove tokens whose cumulative prob exceeds p
    4. Sample from the remaining
    """
    logits = logits / temperature
    probs  = F.softmax(logits, dim=-1)

    # TODO: implement nucleus sampling
    pass


@torch.no_grad()
def topp_generate(model, prompt_ids, max_new_tokens=80, p=0.9, temperature=1.0):
    model.eval()
    ids = torch.tensor([prompt_ids], device=device)
    generated = []
    for _ in range(max_new_tokens):
        logits = model(ids[:, -cfg.max_seq_len:])[:, -1, :].squeeze(0)
        next_token = top_p_sample(logits, p=p, temperature=temperature)
        if next_token is None: break
        generated.append(next_token.item())
        ids = torch.cat([ids, next_token.view(1, 1)], dim=1)
    return generated


print(f'Prompt: "{prompt}"\n')
for p in [0.5, 0.9, 0.99]:
    out = topp_generate(model, prompt_ids, p=p, temperature=0.8)
    print(f'top-p={p}:  "{decode(out)}"')
    print()

## Part E — Using the Library

In [ ]:
from src.generation.sampling import greedy_decode, sample_decode

prompt_tensor = torch.tensor([prompt_ids], device=device)
greedy_out = greedy_decode(model, prompt_tensor, max_new_tokens=60, eos_token_id=None)
print('Library greedy:')
print(decode(greedy_out[0, len(prompt_ids):].tolist()))

sampled_out = sample_decode(model, prompt_tensor, max_new_tokens=60,
                            temperature=0.8, top_k=10, top_p=0.9)
print('\nLibrary sample (T=0.8, top-k=10, top-p=0.9):')
print(decode(sampled_out[0, len(prompt_ids):].tolist()))

## Summary

| Strategy | Deterministic? | Quality | Diversity |
|----------|---------------|---------|----------|
| Greedy | Yes | OK (locally optimal) | Low (repetitive) |
| Temperature | No | Depends on T | Tunable |
| Top-k | No | Good | Moderate |
| Top-p | No | Good | Adaptive |

**Production setting (GPT-4, Gemini):** typically `temperature=0.7–1.0, top_p=0.95, top_k=40` combined.

**Next:** `../part5_integration/13_mini_gemini_project.ipynb` — putting it all together!